In [1]:
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.spatial import cKDTree

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
MODEL_TYPE = 'split_rate_4to1_lycd'
LAND_IMPROVEMENT_RATIO = 4.0
MILLAGE = 13.998  # combined city (0.6317%) + school district (0.7681%) = 1.3998%

LYCD_PATH = Path(
    r'C:\projects\philly_open_avmkit\notebooks\pipeline\data'
    r'\us-pa-philadelphia\out\maps\lycd_by_model_group\all_groups_lycd_per_sqft.csv'
)
DOR_SHP_PATH = Path(
    r'C:\projects\LVTShift\cities\philadelphia\data'
    r'\dor_parcels\Philadelphia_DOR_Parcels_202402.shp'
)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Step 1: Load parcel data

In [2]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
gdf = gpd.read_parquet(PARCEL_PATH)
gdf['parcel_number'] = gdf['parcel_number'].astype(str).str.zfill(9)
print(f'Loaded {len(gdf):,} parcels')

Loaded 579,815 parcels


## Step 2: Derive lot area from DOR parcel polygons (spatial join)

OPA's `total_area` field is zero for ~32K parcels. Instead we compute area from the
Philadelphia DOR parcel shapefile (PASDA 2024) via point-in-polygon spatial join —
matches 99.6% of parcels. The 0.4% remainder get KNN-imputed area in Step 4.

In [3]:
AREA_PATH = DATA_DIR / 'parcel_areas_dor.parquet'

if AREA_PATH.exists():
    dor_areas = pd.read_parquet(AREA_PATH)
    print(f'Loaded DOR area cache: {len(dor_areas):,} rows')
else:
    print('Building DOR area cache via spatial join (one-time, ~30s)...')
    dor = gpd.read_file(DOR_SHP_PATH, columns=['Shape__Are'])
    # DOR shapefile is in EPSG:3857; Shape__Are is in m^2
    pts = gpd.read_parquet(PARCEL_PATH, columns=['parcel_number', 'geometry'])
    pts = pts.to_crs('EPSG:3857')
    joined = gpd.sjoin(pts, dor[['Shape__Are', 'geometry']], how='left', predicate='within')
    joined['dor_area_sqft'] = (joined['Shape__Are'] * 10.7639).round(1)
    dor_areas = joined[['parcel_number', 'dor_area_sqft']].copy()
    dor_areas.to_parquet(AREA_PATH, index=False)
    print(f'  Cached {len(dor_areas):,} rows')

matched = dor_areas['dor_area_sqft'].notna()
print(f'Matched to DOR polygon: {matched.sum():,} ({matched.mean():.1%})')
print(f'Unmatched (needs KNN area imputation): {(~matched).sum():,}')
print(f'area_sqft: median={dor_areas["dor_area_sqft"].median():.0f}, mean={dor_areas["dor_area_sqft"].mean():.0f}')

Loaded DOR area cache: 579,815 rows
Matched to DOR polygon: 577,698 (99.6%)
Unmatched (needs KNN area imputation): 2,117
area_sqft: median=2427, mean=25238


## Step 3: Join LYCD per-sqft values and lot area to parcels

In [4]:
lycd = pd.read_csv(LYCD_PATH, dtype={'key': str})
lycd['key'] = lycd['key'].str.zfill(9)
print(f'LYCD records: {len(lycd):,}  |  model_groups: {lycd["model_group"].value_counts().to_dict()}')

dor_areas['parcel_number'] = dor_areas['parcel_number'].astype(str).str.zfill(9)

# Join DOR lot area
gdf = gdf.merge(dor_areas[['parcel_number', 'dor_area_sqft']], on='parcel_number', how='left')

# Join LYCD psf
gdf = gdf.merge(
    lycd[['key', 'lycd_psf', 'model_group', 'land_pct_used']],
    left_on='parcel_number', right_on='key',
    how='left',
)

lycd_matched = gdf['lycd_psf'].notna()
area_matched = gdf['dor_area_sqft'].notna()
print(f'\nParcels with LYCD psf:  {lycd_matched.sum():,} ({lycd_matched.mean():.1%})')
print(f'Parcels with DOR area:  {area_matched.sum():,} ({area_matched.mean():.1%})')
print(f'Parcels needing LYCD land value imputation: {(~lycd_matched).sum():,}')
print(f'Parcels needing area imputation:            {(~area_matched).sum():,}')

LYCD records: 526,746  |  model_groups: {'residential_sf': 421740, 'residential_mf_small': 37911, 'vacant': 37468, 'residential_mf_large': 15373, 'commercial': 11448, 'residential_apt': 2806}



Parcels with LYCD psf:  524,387 (90.4%)
Parcels with DOR area:  577,736 (99.6%)
Parcels needing LYCD land value imputation: 55,466
Parcels needing area imputation:            2,117


## Step 4: Compute LYCD land values; KNN-impute for unmatched parcels

For LYCD-matched parcels: `lycd_land_value = lycd_psf × dor_area_sqft`.

For the ~2K LYCD-matched parcels that missed the DOR spatial join, area is first
KNN-imputed from the 5 nearest area-matched parcels.

For the ~55K LYCD-unmatched parcels (condo units, recently subdivided lots, etc. whose
individual footprints don't exist in DOR): `lycd_land_value` is KNN-imputed directly
from the 5 nearest LYCD-matched parcels. This avoids multiplying an imputed psf by a
wrong (building-footprint) area.

In [5]:
# Project to EPSG:3857 for Euclidean distance in meters
gdf_m = gdf.to_crs('EPSG:3857')
coords = np.column_stack([gdf_m.geometry.x, gdf_m.geometry.y])
K = 5

def knn_impute_values(values, coords, K):
    values = np.asarray(values, dtype=float)
    has_val = ~np.isnan(values)
    needs_val = np.isnan(values)
    if needs_val.sum() == 0:
        return values
    tree = cKDTree(coords[has_val])
    _, idxs = tree.query(coords[needs_val], k=K, workers=-1)
    out = values.copy()
    out[needs_val] = np.median(values[has_val][idxs], axis=1)
    return out

lycd_matched_mask = gdf['lycd_psf'].notna().values
area_matched_mask = gdf['dor_area_sqft'].notna().values

# Step 4a: KNN-impute area for the ~2K LYCD-matched parcels with no DOR polygon
# (use only area-matched parcels as donors so unmatched-LYCD's bad areas stay isolated)
n_area_missing = (~area_matched_mask).sum()
area_arr = gdf['dor_area_sqft'].values.astype(float)
area_arr = knn_impute_values(area_arr, coords, K)
gdf['dor_area_sqft'] = area_arr
print(f'KNN-imputed dor_area_sqft for {n_area_missing:,} parcels')

# Step 4b: Compute lycd_land_value for LYCD-matched parcels
gdf['lycd_land_value'] = np.where(
    lycd_matched_mask,
    (gdf['lycd_psf'] * gdf['dor_area_sqft']).clip(lower=0),
    np.nan,
)
print(f'Computed lycd_land_value for {lycd_matched_mask.sum():,} LYCD-matched parcels')

# Step 4c: KNN-impute lycd_land_value for LYCD-unmatched parcels
# Uses matched parcels' actual computed land values — no area arithmetic involved
n_lv_missing = (~lycd_matched_mask).sum()
lv_arr = knn_impute_values(gdf['lycd_land_value'].values, coords, K)
gdf['lycd_land_value'] = lv_arr
gdf.loc[~pd.Series(lycd_matched_mask), 'model_group'] = 'imputed'
print(f'KNN-imputed lycd_land_value for {n_lv_missing:,} LYCD-unmatched parcels')

print(f'\nAll parcels have lycd_land_value: {np.isnan(gdf["lycd_land_value"].values).sum() == 0}')

KNN-imputed dor_area_sqft for 2,117 parcels
Computed lycd_land_value for 524,387 LYCD-matched parcels
KNN-imputed lycd_land_value for 55,466 LYCD-unmatched parcels

All parcels have lycd_land_value: True


## Step 5: Summarize LYCD land base

In [6]:
total_lycd_land = gdf['lycd_land_value'].sum()
total_opa_land = gdf['taxable_land'].sum()
print(f'Total LYCD land base:    ${total_lycd_land/1e9:.2f}B')
print(f'Total OPA taxable land:  ${total_opa_land/1e9:.2f}B')
print(f'Ratio LYCD/OPA:          {total_lycd_land/total_opa_land:.2f}x')
print()
print('LYCD land value by model_group (median):')
print(gdf.groupby('model_group')['lycd_land_value'].median().sort_values(ascending=False).to_string())

Total LYCD land base:    $117.60B
Total OPA taxable land:  $36.63B
Ratio LYCD/OPA:          3.21x

LYCD land value by model_group (median):
model_group
residential_apt         392345.590446
imputed                 227495.435685
commercial              148184.972763
residential_mf_small    115231.648933
residential_mf_large     72896.276243
residential_sf           71141.799134
vacant                   60965.876347


## Step 6: Categorize parcels (same overrides as OPA model)

In [7]:
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement -> Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: abated improved parcels
GENUINE_VACANT_CODES = {'6', '12', '13'}
abated_mask = (
    (gdf['taxable_building'] <= 0) &
    (gdf['taxable_land'] > 0) &
    (~gdf['category_code'].isin(GENUINE_VACANT_CODES))
)
gdf.loc[abated_mask, 'PROPERTY_CATEGORY'] = 'Abated / Construction Exemption'

# Override 3: OPA-vacant with nonzero building value
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: fully exempt parcels
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

Total parcels: 579,853
Fully exempt: 44,134  |  Abated: 30,129  |  Improved vacant: 1,485  |  Taxable: 535,719

Property category distribution:
PROPERTY_CATEGORY
Single Family Residential                  408141
Small Multi-Family (2-4 units)              37944
Abated / Construction Exemption             30129
Vacant Land                                 27927
Single Family Residential — Exempt          27059
Mixed Use                                   13399
Vacant Land — Exempt                        11860
Commercial                                   8521
Industrial                                   3489
Commercial — Exempt                          3280
Large Multi-Family (5+ units)                2547
Improved Vacant Land                         1485
Small Multi-Family (2-4 units) — Exempt      1114
Other Residential                            1068
Office / Commercial Condo                     805
Large Multi-Family (5+ units) — Exempt        300
Industrial — Exempt                   

## Step 7: Current tax (OPA taxable values — revenue baseline)

In [8]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_only_rate = 6.317  # mills
city_revenue = gdf['taxable_total'].mul(city_only_rate / 1000).sum()
gap_pct = (city_revenue / 795_796_126 - 1) * 100

print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion (0.6317%):    ${city_revenue:,.0f}')
print(f'FY2024 city-only actuals (approx):      $795,796,126')
print(f'City portion gap: {gap_pct:+.1f}%  (target: within ±5%)')
assert abs(gap_pct) < 10.0, f'City gap {gap_pct:.1f}% exceeds 10%'

Modeled combined levy (city + school):  $1,851,235,135
Implied city-only portion (0.6317%):    $835,423,086
FY2024 city-only actuals (approx):      $795,796,126
City portion gap: +5.0%  (target: within ±5%)


## Step 8: Build LYCD reform base

Land value: LYCD-derived (`lycd_land_value` — either psf × DOR area for matched parcels,
or KNN-imputed land value for unmatched parcels).
Building value: OPA `taxable_building` — preserves existing Homestead and abatement exemption structure.

For abated parcels (OPA shows zero building): impute building = 4 × LYCD land value
(restoring OPA's implicit 20% land ratio applied to the LYCD-estimated land base).

In [9]:
IMPUTED_BLDG_MULTIPLIER = 4.0

gdf['model_land'] = gdf['lycd_land_value'].clip(lower=0)
gdf['model_building'] = gdf['taxable_building'].clip(lower=0)

abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
gdf.loc[abated, 'model_building'] = gdf.loc[abated, 'model_land'] * IMPUTED_BLDG_MULTIPLIER

n_imputed = abated.sum()
imputed_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels with imputed building value: {n_imputed:,}')
print(f'Total imputed building base added to reform:  ${imputed_bldg_total/1e9:.2f}B')
print()
print(f'Reform land base:          ${gdf["model_land"].sum()/1e9:.2f}B')
print(f'Reform improvement base:   ${gdf["model_building"].sum()/1e9:.2f}B')
print(f'OPA taxable land base:     ${gdf["taxable_land"].sum()/1e9:.2f}B')
print(f'OPA taxable building base: ${gdf["taxable_building"].sum()/1e9:.2f}B')

Abated parcels with imputed building value: 30,129
Total imputed building base added to reform:  $29.99B

Reform land base:          $117.60B
Reform improvement base:   $125.61B
OPA taxable land base:     $36.63B
OPA taxable building base: $95.62B


## Step 9: Revenue-neutral split-rate model (4:1 land:improvement)

In [10]:
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)')

Land millage:        14.0202 mills
Improvement millage: 3.5051 mills
Revenue check:       $1,851,235,135 (target: $1,851,235,135)




Philadelphia — 4:1 Split-Rate Tax Impact (LYCD Land Values)
                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 408141    $114,927,310       11.1%       $282        $-384   27.6%     -23.3%            22.7%            64.6%
         Small Multi-Family (2-4 units)  37944    $-86,900,249      -39.7%    $-2,290      $-1,109  -22.1%     -34.3%            10.7%            81.2%
        Abated / Construction Exemption  30129    $166,404,513      379.9%     $5,523       $1,301 2102.7%     468.7%            96.2%             2.4%
                            Vacant Land  27927     $61,711,181      181.2%     $2,210         $383  473.5%     102.7%            83.0%            11.5%
     Single Family Residential — Exempt  27059              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
                           

## Step 10: Census join

In [11]:
import concurrent.futures

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


## Step 11: Export and visualize

In [12]:
out_df = save_standard_export(
    df=gdf,
    city=f'{CITY_NAME}_lycd',
    output_path=f'../../analysis/data/{CITY_NAME}_lycd.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
)

from lvt.viz import create_city_report
create_city_report(out_df, f'{CITY_NAME}_lycd', show=False)
print('Done.')

  [warn] philadelphia_lycd: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia_lycd: 579,853 rows → ../../analysis/data/philadelphia_lycd.csv  [model: split_rate_4to1_lycd]


Done.
